# Data Cleaning & Preparation Notebook

## Research Question
**Is biodiversity loss in beef-exporting countries associated with the volume of beef exports to the United Kingdom?**

## Data Sources
| Dataset | File | Description                                                                                                             |
|---------|------|-------------------------------------------------------------------------------------------------------------------------|
| BIO | `bio-raw.csv` | Biodiversity Intactness Index (BII) per country, scenario, and year. Historical: 1970?2014; SSP projections: 2015?2050. |
| FAOSTAT | `FAOSTAT_data_en_3-31-2026.csv` | Annual bilateral trade: beef & veal export quantities to the UK, by reporting country, 2001?2024.                       |

## Variables
| Variable | Role | Description |
|----------|------|-------------|
| `bii` | Predictor (raw) | Biodiversity Intactness Index (0?1; 1 = fully intact) |
| `bii_loss` | Predictor (derived) | 1 ? BII (higher = more biodiversity loss) |
| `uk_import_qty_t` | Outcome (raw) | Beef export quantity to UK (tonnes) |
| `log_uk_import_qty_t` | Outcome (transformed) | log(1 + uk_import_qty_t) ? corrects right-skew |

## Timescale Rationale
- Historical BIO data runs to **2014**; FAOSTAT starts at **2001**.
- Overlap for fully observed data: **2001?2014** (14 years).
- To extend to 2024 we stitch in **SSP2** ("middle of the road") BIO projections for 2015?2024.

## Outputs
All saved to `data/clean/`:
| File | Description |
|------|-------------|
| `bio_clean.csv` | BII per country ? year ? scenario, ISO3 extracted |
| `fao_clean.csv` | FAOSTAT tidy: iso3, country, year, uk_import_qty_t |
| `panel_2001_2014.csv` | Panel (country ? year), 2001?2014, historical BIO only |
| `crosssection_avg.csv` | Cross-sectional country averages, 2001?2014 |
| `panel_extended_2001_2024.csv` | Extended panel: historical + SSP2 BIO, 2001?2024 |

---
## 0 ? Setup

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 120)

# ?? Paths ??????????????????????????????????????????????????????????????????
# This notebook lives in data/, so paths are relative to that folder.
DATA = os.path.dirname(os.path.abspath("__file__"))  # data/
OUT  = os.path.join(DATA, "clean")
os.makedirs(OUT, exist_ok=True)

BIO_PATH = os.path.join(DATA, "bio-raw.csv")
FAO_PATH = os.path.join(DATA, "FAOSTAT_data_en_3-31-2026.csv")

# ?? Parameters ?????????????????????????????????????????????????????????????
# Minimum total exports to UK (tonnes) across all years to keep a country.
MIN_TOTAL_TRADE_T = 10

# Importing country we focus on.
IMPORT_COUNTRY = "United Kingdom of Great Britain and Northern Ireland"

# SSP scenario used to extend BIO beyond 2014.
EXTENSION_SCENARIO = "ssp2rcp4p5messageglobiom"  # "middle of the road"


### ISO-3 Country-Code Helper

We use a comprehensive built-in lookup table that covers every FAOSTAT
reporter-country name (including FAO-specific spellings like "Trkiye" and
"China, mainland").  `pycountry` is used only as a fallback for any names
not already in the table.

> **Why not rely on `pycountry` alone?**  Two reasons:
> 1. `pycountry` may not be installed in the Jupyter kernel (the old code
>    caught `ImportError` silently via `except Exception: pass`, causing
>    *all* non-override lookups to return `None`).
> 2. Some FAO names ("China, mainland", "Netherlands (Kingdom of the)",
>    "Trkiye") have no match in the ISO standard even with fuzzy search.

In [ ]:
# ?? Complete FAOSTAT-name ? ISO3 lookup ????????????????????????????????????
# Covers every Reporter Countries value in the FAOSTAT beef-trade extract.
FAOSTAT_ISO3 = {
    "Argentina":                    "ARG",
    "Australia":                    "AUS",
    "Austria":                      "AUT",
    "Bahrain":                      "BHR",
    "Belgium":                      "BEL",
    "Botswana":                     "BWA",
    "Brazil":                       "BRA",
    "Brunei Darussalam":            "BRN",
    "Bulgaria":                     "BGR",
    "China, mainland":              "CHN",
    "China, Taiwan Province of":    "TWN",
    "Colombia":                     "COL",
    "Croatia":                      "HRV",
    "Cyprus":                       "CYP",
    "Czechia":                      "CZE",
    "Denmark":                      "DNK",
    "Dominican Republic":           "DOM",
    "Estonia":                      "EST",
    "Finland":                      "FIN",
    "France":                       "FRA",
    "Germany":                      "DEU",
    "Ghana":                        "GHA",
    "Greece":                       "GRC",
    "Hungary":                      "HUN",
    "Iceland":                      "ISL",
    "India":                        "IND",
    "Indonesia":                    "IDN",
    "Ireland":                      "IRL",
    "Israel":                       "ISR",
    "Italy":                        "ITA",
    "Jamaica":                      "JAM",
    "Kyrgyzstan":                   "KGZ",
    "Latvia":                       "LVA",
    "Lebanon":                      "LBN",
    "Lithuania":                    "LTU",
    "Luxembourg":                   "LUX",
    "Malta":                        "MLT",
    "Mozambique":                   "MOZ",
    "Namibia":                      "NAM",
    "Netherlands (Kingdom of the)": "NLD",
    "New Zealand":                  "NZL",
    "Norway":                       "NOR",
    "Philippines":                  "PHL",
    "Poland":                       "POL",
    "Portugal":                     "PRT",
    "Republic of Korea":            "KOR",
    "Romania":                      "ROU",
    "Russian Federation":           "RUS",
    "Singapore":                    "SGP",
    "Slovakia":                     "SVK",
    "Slovenia":                     "SVN",
    "South Africa":                 "ZAF",
    "Spain":                        "ESP",
    "Sweden":                       "SWE",
    "Switzerland":                  "CHE",
    "Trinidad and Tobago":          "TTO",
    "Trkiye":                       "TUR",
    "T?rkiye":                      "TUR",
    "Ukraine":                      "UKR",
    "United Arab Emirates":         "ARE",
    "United States of America":     "USA",
    "Uruguay":                      "URY",
    "Viet Nam":                     "VNM",
    "Zimbabwe":                     "ZWE",
}

# Track whether pycountry is available (checked once, not per-row).
try:
    import pycountry as _pycountry
    _HAS_PYCOUNTRY = True
except ImportError:
    _HAS_PYCOUNTRY = False
    print("NOTE: pycountry is not installed ? using built-in FAOSTAT_ISO3 table only.")


def name_to_iso3(name: str):
    """Return ISO 3166-1 alpha-3 for a country name string.

    Resolution order:
      1. Built-in FAOSTAT_ISO3 dictionary  (fast, no dependencies)
      2. pycountry exact match             (if installed)
      3. pycountry fuzzy search            (if installed)
    """
    # 1) Built-in table ? covers all known FAOSTAT names
    if name in FAOSTAT_ISO3:
        return FAOSTAT_ISO3[name]

    # 2) pycountry fallback for any new / unknown names
    if _HAS_PYCOUNTRY:
        try:
            result = _pycountry.countries.get(name=name)
            if result:
                return result.alpha_3
            hits = _pycountry.countries.search_fuzzy(name)
            if hits:
                return hits[0].alpha_3
        except LookupError:
            pass

    return None

---
# Part A ? Biodiversity (BIO) Data
## 1 ? Load & Explore Raw BIO Data

The BIO dataset contains multiple indicators (BII, land-use shares, etc.).
Since our research question focuses on **biodiversity loss**, we retain only
the **Biodiversity Intactness Index (`bii`)** and derive `bii_loss = 1 ? bii`.
All other indicators (pastureland, crops, highintensityag, etc.) are dropped*.

In [ ]:
bio_raw = pd.read_csv(BIO_PATH, low_memory=False)

print(f"Shape: {bio_raw.shape}")
print(f"Columns: {list(bio_raw.columns)}")
print(f"Year range: {bio_raw['year'].min()} ? {bio_raw['year'].max()}  (dtype: {bio_raw['year'].dtype})")
print(f"Scenarios: {bio_raw['scenario'].unique()}")
print(f"Variables (indicators): {bio_raw['variable'].unique()}")
bio_raw.head()

### 1.1 ? Validate: null counts & data quality

In [ ]:
print("Null counts per column:")
print(bio_raw.isna().sum())
print(f"\nRows with null area_code: {bio_raw['area_code'].isna().sum():,}")
print(f"Rows with null value:     {bio_raw['value'].isna().sum():,}")
print(f"Total rows:               {len(bio_raw):,}")

### 1.2 ? Understand the `area_code` structure
Each `area_code` is a dash-separated hierarchy (continent-region-?-ISO3).
The **last segment** is the ISO 3166-1 alpha-3 country code ? that's what we need.

In [ ]:
print("Sample area_codes:")
print(bio_raw["area_code"].dropna().unique()[:10])

## 2 ? Clean BIO Data

Steps:
1. **Drop rows** with null `area_code` (no country identity ? useless).
2. **Extract ISO3** from the last segment of `area_code`.
3. **Keep only valid 3-letter codes** (drop supra-national aggregates like numeric codes).
4. **Filter to `bii` indicator only** ? the sole predictor for our research question.
5. **Drop unneeded columns**: `_id` (database artefact), `lower_uncertainty` / `upper_uncertainty` (all NA for BII).
6. **Drop rows with null `value`** (no measurement ? can't model).
7. **Pivot wide** so `bii` becomes its own column.
8. **Derive `bii_loss`** = 1 ? BII (higher = more degraded, easier to interpret as a predictor).
9. **Drop countries with no BII data** ? small islands/territories that lack BII and don't export beef to the UK.

In [ ]:
# Step 2.1 ? Drop rows with no area code
bio = bio_raw.dropna(subset=["area_code"]).copy()
print(f"After dropping null area_code: {bio.shape[0]:,} rows  (removed {len(bio_raw) - len(bio):,})")

# Step 2.2 ? Extract ISO3 from the last segment of area_code
bio["iso3"] = bio["area_code"].str.split("-").str[-1].str.upper()

# Step 2.3 ? Keep only standard 3-letter ISO3 codes
bio = bio[bio["iso3"].str.match(r"^[A-Z]{3}$", na=False)]
print(f"After filtering to valid ISO3 codes: {bio.shape[0]:,} rows  |  {bio['iso3'].nunique()} countries")

# Step 2.4 ? Keep only the BII indicator (our sole biodiversity predictor)
bio = bio[bio["variable"] == "bii"].copy()
print(f"After filtering to BII indicator: {bio.shape[0]:,} rows")

# Step 2.5 ? Drop columns not needed for modelling
bio.drop(columns=["_id", "area_code", "lower_uncertainty", "upper_uncertainty", "variable"], inplace=True)

# Step 2.6 ? Drop rows with missing BII value
before = len(bio)
bio.dropna(subset=["value"], inplace=True)
print(f"After dropping null values: {bio.shape[0]:,} rows  (removed {before - len(bio):,})")

# Step 2.7 ? Rename value column
bio.rename(columns={"value": "bii"}, inplace=True)

# Step 2.8 ? Derive bii_loss = 1 - bii  (higher = more degraded)
bio["bii_loss"] = 1 - bio["bii"]

INDICATOR_COLS = ["bii", "bii_loss"]

print(f"\nCleaned BIO: {bio.shape}")
print(f"  Scenarios:  {bio['scenario'].nunique()} ? {sorted(bio['scenario'].unique())}")
print(f"  Countries:  {bio['iso3'].nunique()}")
print(f"  Years:      {bio['year'].min()} ? {bio['year'].max()}")
bio.head()

In [ ]:
# Data is already in wide format (one row per iso3 ? year ? scenario)
# since we filtered to the single BII indicator above.

bio_wide = bio.copy()

print(f"BIO shape: {bio_wide.shape}")
print(f"Columns: {list(bio_wide.columns)}")
print(f"Countries: {bio_wide['iso3'].nunique()}")
bio_wide.head(10)

### 2.1 ? Validate: BIO cleaning results

In [ ]:
print("Null counts per column after cleaning:")
print(bio_wide[INDICATOR_COLS].isna().sum())

print(f"\nHistorical rows: {(bio_wide['scenario'] == 'historical').sum():,}")
print(f"SSP2 rows:       {(bio_wide['scenario'] == EXTENSION_SCENARIO).sum():,}")

print("\nDescriptive stats ? historical scenario:")
bio_wide[bio_wide["scenario"] == "historical"][INDICATOR_COLS].describe().round(4)

In [ ]:
# Save cleaned BIO
bio_wide.to_csv(os.path.join(OUT, "bio_clean.csv"), index=False)
print(f"Saved ? data/clean/bio_clean.csv  ({bio_wide.shape[0]:,} rows, {bio_wide.shape[1]} cols)")

### 2.2 ? Historical BIO slice (for merging later)
We isolate the **historical** scenario ? these are observed data (not model projections).
This is the most reliable subset for analysis.

In [ ]:
bio_hist = (
    bio_wide[bio_wide["scenario"] == "historical"]
    .drop(columns=["scenario"])
    .copy()
)
print(f"Historical BIO: {bio_hist.shape[0]:,} rows")
print(f"Year range: {bio_hist['year'].min()} ? {bio_hist['year'].max()}")
print(f"Countries:  {bio_hist['iso3'].nunique()}")
bio_hist.head()

---
# Part B ? FAOSTAT Trade Data
## 3 ? Load & Explore Raw FAOSTAT Data

> **Note:** After cleaning, only ~32 countries remain with non-negligible
> UK beef exports. Trade volumes are heavily concentrated in a few countries
> (Ireland, Brazil, Netherlands) ? log-transformation is used to reduce
> skewness and the influence of dominant exporters.

In [ ]:
fao_raw = pd.read_csv(FAO_PATH, encoding="utf-8-sig")

print(f"Shape: {fao_raw.shape}")
print(f"Columns: {list(fao_raw.columns)}")
print(f"Year range: {fao_raw['Year'].min()} ? {fao_raw['Year'].max()}")
print(f"Partner countries: {fao_raw['Partner Countries'].unique()}")
print(f"Items: {fao_raw['Item'].unique()}")
print(f"Elements: {fao_raw['Element'].unique()}")
print(f"Reporter countries (exporters): {fao_raw['Reporter Countries'].nunique()}")
fao_raw.head()

### 3.1 ? Validate: FAOSTAT data quality

In [ ]:
print("Null counts:")
print(fao_raw.isna().sum())
print(f"\nFlag descriptions (data quality):")
print(fao_raw["Flag Description"].value_counts())
print(f"\nAll rows import to: {fao_raw['Partner Countries'].unique()}")

## 4 ? Clean FAOSTAT Data

Steps:
1. **Filter** for the target importing country (United Kingdom).
2. **Select & rename** columns for clarity.
3. **Drop zero-trade rows** (no useful information).
4. **Map country names ? ISO3** using built-in lookup + `pycountry` fallback.
5. **Remove negligible trade partners** (< 10 tonnes total across all years).
6. **Log-transform** the outcome variable (trade volumes are heavily right-skewed).

In [ ]:
# Step 4.1 ? Filter for UK imports
fao = fao_raw[fao_raw["Partner Countries"] == IMPORT_COUNTRY].copy()
print(f"After filtering to UK imports: {fao.shape[0]} rows")

# Step 4.2 ? Select and rename columns
fao = fao[[
    "Reporter Countries", "Partner Countries",
    "Item", "Element", "Year", "Unit", "Value", "Flag Description",
]].rename(columns={
    "Reporter Countries": "country",
    "Partner Countries":  "import_country",
    "Item":               "commodity",
    "Element":            "measure",
    "Year":               "year",
    "Unit":               "unit",
    "Value":              "uk_import_qty_t",
    "Flag Description":   "data_quality",
})

# Step 4.3 ? Drop zero-trade rows
before = len(fao)
fao = fao[fao["uk_import_qty_t"] > 0].copy()
print(f"After dropping zero-trade rows: {fao.shape[0]} rows  (removed {before - len(fao)})")

# Step 4.4 ? Map country names to ISO3
fao["iso3"] = fao["country"].map(name_to_iso3)

unmapped = fao[fao["iso3"].isna()]["country"].unique()
if len(unmapped):
    print(f"WARNING: Could not map to ISO3: {unmapped}")
else:
    print(f"All countries mapped to ISO3 ?  ({fao['iso3'].nunique()} unique countries, {len(fao)} rows)")

# Show a sample from different countries (head() only shows Argentina because it's first alphabetically)
fao.groupby("country").head(1).head(10)

In [ ]:
# Step 4.5 ? Remove negligible trade partners
total_by_country = fao.groupby("country")["uk_import_qty_t"].sum()
keep_countries   = total_by_country[total_by_country >= MIN_TOTAL_TRADE_T].index
removed_countries = set(fao["country"].unique()) - set(keep_countries)

fao = fao[fao["country"].isin(keep_countries)].copy()
print(f"Removed {len(removed_countries)} countries with < {MIN_TOTAL_TRADE_T}t total trade: {removed_countries}")
print(f"Remaining: {fao['country'].nunique()} exporting countries")

# Sort and reset index
fao.sort_values(["iso3", "year"], inplace=True)
fao.reset_index(drop=True, inplace=True)

# Step 4.6 ? Log-transform outcome (handles right-skew)
fao["log_uk_import_qty_t"] = np.log1p(fao["uk_import_qty_t"])

print(f"\nCleaned FAOSTAT: {fao.shape}")
print(f"Year range: {fao['year'].min()} ? {fao['year'].max()}")
print(f"Unique exporters: {fao['iso3'].nunique()}")

### 4.1 ? Validate: top exporters to the UK

In [ ]:
top_exporters = (
    fao.groupby(["iso3", "country"])["uk_import_qty_t"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
)
print("Top 15 beef exporters to the UK (total tonnes, 2001-2024):")
print(top_exporters.to_string())

In [ ]:
# Save cleaned FAOSTAT
fao.to_csv(os.path.join(OUT, "fao_clean.csv"), index=False)
print(f"Saved -> data/clean/fao_clean.csv  ({fao.shape[0]:,} rows)")

---
# Part C ? Merge & Build Analysis-Ready Datasets

We join BIO indicators with FAOSTAT trade data on `(iso3, year)` to create
panel and cross-sectional datasets ready for regression.

## 5 ? Panel-Building Helper

In [ ]:
def build_panel(bio_slice, fao_df, label):
    # Aggregate FAOSTAT by (iso3, year, country) in case of multiple commodity rows
    fao_agg = (
        fao_df.groupby(["iso3", "year", "country"], as_index=False)
              .agg(uk_import_qty_t=("uk_import_qty_t", "sum"))
    )
    fao_agg["log_uk_import_qty_t"] = np.log1p(fao_agg["uk_import_qty_t"])

    # Inner join: only country-years present in BOTH datasets
    merged = bio_slice.merge(fao_agg, on=["iso3", "year"], how="inner")
    merged.sort_values(["iso3", "year"], inplace=True)
    merged.reset_index(drop=True, inplace=True)

    # Drop rows with NaN in any model column
    model_cols = INDICATOR_COLS + ["uk_import_qty_t"]
    before = len(merged)
    merged.dropna(subset=model_cols, inplace=True)
    dropped = before - len(merged)

    # Reorder columns: identifiers -> outcome -> predictors
    id_cols = ["iso3", "country", "year"]
    y_cols  = ["uk_import_qty_t", "log_uk_import_qty_t"]
    merged  = merged[id_cols + y_cols + INDICATOR_COLS]

    n_countries = merged["iso3"].nunique()
    n_years     = merged["year"].nunique()
    print(f"\n{label}")
    print(f"  Shape: {merged.shape}  |  {n_countries} countries x {n_years} years")
    if dropped:
        print(f"  Dropped {dropped} rows with NaN in model columns")
    return merged

## 6 ? Panel 2001?2014 (Historical BIO Only)
This is the **primary analysis dataset** ? both BIO and FAOSTAT are fully observed (no projections).

In [ ]:
panel_hist = build_panel(
    bio_hist[bio_hist["year"].between(2001, 2014)],
    fao[fao["year"].between(2001, 2014)],
    "Panel 2001-2014 (historical BIO)",
)

panel_hist.to_csv(os.path.join(OUT, "panel_2001_2014.csv"), index=False)
print(f"Saved -> data/clean/panel_2001_2014.csv")
panel_hist.head(10)

### 6.1 ? Validate: panel coverage

In [ ]:
print(f"Countries in panel: {sorted(panel_hist['iso3'].unique())}")
print(f"Years in panel:     {sorted(panel_hist['year'].unique())}")
print(f"\nObservations per year:")
print(panel_hist.groupby("year")["iso3"].count())

## 7 ? Cross-Sectional Dataset (Country Averages 2001?2014)
One row per country ? averages of all variables over 2001?2014.


In [ ]:
cs = (
    panel_hist
    .groupby(["iso3", "country"])
    .agg(
        years_observed=("year", "count"),
        **{col: (col, "mean") for col in INDICATOR_COLS},
        uk_import_qty_t=("uk_import_qty_t", "mean"),
        log_uk_import_qty_t=("log_uk_import_qty_t", "mean"),
    )
    .reset_index()
    .sort_values("uk_import_qty_t", ascending=False)
    .reset_index(drop=True)
)

print(f"Cross-sectional shape: {cs.shape}  ({cs.shape[0]} countries)")
cs.to_csv(os.path.join(OUT, "crosssection_avg.csv"), index=False)
print(f"Saved -> data/clean/crosssection_avg.csv")

# Display sorted by UK import volume
display_cols = ["iso3", "country", "uk_import_qty_t", "bii", "bii_loss"]
cs[[c for c in display_cols if c in cs.columns]]

## 8 ? Extended Panel 2001?2024 (Historical + SSP2 Projections)

To extend coverage beyond 2014 we stitch in **SSP2** ("middle of the road") BIO projections for 2015?2024.

**2015?2024 BIO values are modelled, not observed**

In [ ]:
# SSP2 BIO slice for 2015-2024
bio_ssp2 = (
    bio_wide[
        (bio_wide["scenario"] == EXTENSION_SCENARIO) &
        (bio_wide["year"].between(2015, 2024))
    ]
    .drop(columns=["scenario"])
    .copy()
)
print(f"SSP2 BIO slice: {bio_ssp2.shape[0]:,} rows  |  years {bio_ssp2['year'].min()}-{bio_ssp2['year'].max()}")

# Combine: historical 2001-2014 + SSP2 2015-2024
bio_combined = pd.concat([
    bio_hist[bio_hist["year"].between(2001, 2014)],
    bio_ssp2,
], ignore_index=True)

panel_ext = build_panel(bio_combined, fao, "Extended panel 2001-2024 (historical + SSP2)")

# Attach bio_source flag
bio_src = bio_combined[["iso3", "year"]].copy()
bio_src["bio_source"] = np.where(bio_combined["year"] <= 2014, "historical", "ssp2_projected")
panel_ext = panel_ext.merge(bio_src.drop_duplicates(), on=["iso3", "year"], how="left")

panel_ext.to_csv(os.path.join(OUT, "panel_extended_2001_2024.csv"), index=False)
print(f"Saved -> data/clean/panel_extended_2001_2024.csv")
panel_ext.head(10)

---
# Part D ? Validation & Summary Statistics
## 9 ? Descriptive Statistics (Cross-Sectional)

In [ ]:
desc_cols = ["uk_import_qty_t", "log_uk_import_qty_t"] + INDICATOR_COLS
cs[desc_cols].describe().round(4)

### 9.1 ? Correlations with log UK import quantity
We use the **log-transformed** outcome because raw trade volumes are heavily right-skewed.

In [ ]:
corr = (
    cs[desc_cols]
    .corr()["log_uk_import_qty_t"]
    .drop(["uk_import_qty_t", "log_uk_import_qty_t"])
    .sort_values(key=abs, ascending=False)
)
print("Pearson correlations with log_uk_import_qty_t:")
print(corr.round(4).to_string())

## 10 ? Quick OLS Sanity Check (Cross-Sectional)

A preliminary OLS to verify data is usable before moving to full modelling.

- **Model**: `log_import ~ bii_loss`

This tests whether countries with more biodiversity loss export more beef to the UK.
We use **HC3 robust standard errors** to guard against heteroskedasticity from
dominant exporters (Ireland, Brazil).

In [ ]:
try:
    import statsmodels.api as sm

    y = cs["log_uk_import_qty_t"].astype(float)

    # -- OLS: bii_loss ? log UK beef imports -----------------------------------
    print("OLS: log_uk_import_qty_t ~ bii_loss")
    X = sm.add_constant(cs[["bii_loss"]].astype(float))

    # Standard OLS
    m = sm.OLS(y, X).fit()
    print("\n--- Standard errors ---")
    print(m.summary2().tables[1].round(4))
    print(f"R-sq = {m.rsquared:.4f}  |  Adj-R-sq = {m.rsquared_adj:.4f}  |  N = {int(m.nobs)}")

    # Robust (HC3) standard errors
    m_robust = sm.OLS(y, X).fit(cov_type="HC3")
    print("\n--- Robust (HC3) standard errors ---")
    print(m_robust.summary2().tables[1].round(4))
    print(f"R-sq = {m_robust.rsquared:.4f}  |  Adj-R-sq = {m_robust.rsquared_adj:.4f}  |  N = {int(m_robust.nobs)}")

except ImportError:
    print("statsmodels not installed -- run:  pip install statsmodels")

---
## Summary

**Outcome variable (Y):** `uk_import_qty_t` / `log_uk_import_qty_t`

**Predictor (X):** `bii_loss` (= 1 ? BII; higher = more biodiversity loss)

---

### Next Steps for Modelling

1. **Use the extended panel (2001?2024)** with fixed/random effects for more statistical power.
2. **Apply robust standard errors** to guard against heteroskedasticity from dominant exporters.
3. **Run influence diagnostics** (Cook's distance) to check whether results are driven by Ireland or Brazil.
